# <center>**Enhancing LEM-X Imaging with the IROS Reconstruction Pipeline**<center>

## <center>**Sky Reconstruction Efficiency**<center>

In [1]:
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
import darksun as ds
from darksun.data import Log, DataLoader, CatalogueLoader

from IROSrec.handle import config_dirpaths
import imgmaker as mgm
from imgmaker.fns import CameraUnitMap

In [2]:
MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"

SKYFIELD: str = "GalacticCentre"
# SKYFIELD: str = "IROSDummy"
DATA_FITS: str = "baseline_2-50keV_1ks"

RUN_ID: str = 'GC_upx5upy1'

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "reconstructed"

E_min: float = 2.0  # [keV]
E_max: float = 50.0  # [keV]

UP_X, UP_Y = 5, 1

In [3]:
MASK_PATH, SIMUL_DATA_PATH, SAVE_PATH = config_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
    runID=RUN_ID,
)
OUT_RESULTS_PATH = mgm.config_savedata_to()

wfm: CodedMaskCamera = codedmask(MASK_PATH, UP_X, UP_Y)

filepaths: dict[str, dict[str, Path]] = simulation_files(SIMUL_DATA_PATH)
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max)
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET], E_min=E_min, E_max=E_max)
catB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

logA, logB = ds.load_database(f"{SAVE_PATH}/IROS_sources_db.fits")

# Loading data...
# Loading completed!


### <center>**Benchmark Tables**<center>

In [4]:
import re

def adjust_Tabfrmt(txt: str) -> str:
    # insert \hline instead of rules (journal guidelines)
    for rule in ('toprule', 'midrule', 'bottomrule'):
        txt = txt.replace(rule, 'hline')
    # shift caption and label at the end (journal guidelines)
    pattern = r"(\\begin\{table\}.*?)(\\caption\{.*?\})\s*(\\label\{.*?\})\s*(\\begin\{tabular\}.*?\\end\{tabular\})"
    replacement = r"\1\4\n\2\n\3"
    txt = re.sub(pattern, replacement, txt, flags=re.DOTALL)
    # convert to onecolumn
    txt = txt.replace('table', 'table*')
    return txt

def sort_by(df: pd.DataFrame, key: str, **kwargs: Any) -> pd.DataFrame:
    """Sort DataFrame wrt input column key."""
    return df.sort_values(by=[key], ascending=False, ignore_index=True, **kwargs)

In [5]:
from numpy.typing import NDArray

def gather_cam_data(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
    varmap: NDArray,
) -> pd.DataFrame:
    """
    Gathers single camera data from IROS reconstruction database.
    """
    ids = np.array([src.upper() for src in log.log['ID']])
    theta_res_x, theta_res_y = mgm.get_angularcoords_residues(
        log, catalogue, sdl, camera,
    )
    cts = np.array(log.log['fluence'])
    true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
    n, m = varmap.shape
    boxsize = (80, 200)
    srows, scols = (
        slice((n - 1) // 2 - camera.upscale_f.y * boxsize[0], (n - 1) // 2 + camera.upscale_f.y * boxsize[0] + 1),
        slice((m - 1) // 2 - camera.upscale_f.x * boxsize[1], (m - 1) // 2 + camera.upscale_f.x * boxsize[1] + 1),
    )
    rmse = np.sqrt(np.mean(varmap[srows, scols]))
    dmap = {
        log.name: {
            'Source': ids,
            'DthetaX': theta_res_x,
            'DthetaY': theta_res_y,
            'IROS_cts': cts,
            'True_cts': true_cts,
            'Dcts': (cts - true_cts) / np.sqrt(true_cts),
            'Dcts_var': (cts - true_cts) / rmse,
            'SNR': np.array(log.log['snr']),
            # 'thetaX [deg]': np.array(log.log['angle_x']),
            # 'thetaY [deg]': np.array(log.log['angle_y']),
        }
    }
    return pd.DataFrame(dmap)

def get_joint_tab(
    data_camA: pd.DataFrame,
    data_camB: pd.DataFrame,
    unitmap: CameraUnitMap,
) -> pd.DataFrame:
    """Generates a Dataframe with output data from both cameras."""
    compose: Callable = lambda a, b: np.sqrt(a ** 2 + b ** 2)
    dmap = {
        'Source': np.array(data_camA.CAM1A['Source'])[unitmap.idx_a],

        'DthetaX_A': np.array(data_camA.CAM1A['DthetaX'])[unitmap.idx_a],
        'DthetaY_A': np.array(data_camA.CAM1A['DthetaY'])[unitmap.idx_a],
        'TrueCts_A': np.array(data_camA.CAM1A['True_cts'])[unitmap.idx_a],
        'ReconstrCts_A': np.array(data_camA.CAM1A['IROS_cts'])[unitmap.idx_a],
        'Dcts_A': np.array(data_camA.CAM1A['Dcts'])[unitmap.idx_a],
        'Dcts_A_var': np.array(data_camA.CAM1A['Dcts_var'])[unitmap.idx_a],

        'DthetaX_B': np.array(data_camB.CAM1B['DthetaX'])[unitmap.idx_b],
        'DthetaY_B': np.array(data_camB.CAM1B['DthetaY'])[unitmap.idx_b],
        'TrueCts_B': np.array(data_camB.CAM1B['True_cts'])[unitmap.idx_b],
        'ReconstrCts_B': np.array(data_camB.CAM1B['IROS_cts'])[unitmap.idx_b],
        'Dcts_B': np.array(data_camB.CAM1B['Dcts'])[unitmap.idx_b],
        'Dcts_B_var': np.array(data_camB.CAM1B['Dcts_var'])[unitmap.idx_b],

        'SNR': compose(
            np.array(data_camA.CAM1A['SNR'])[unitmap.idx_a],
            np.array(data_camB.CAM1B['SNR'])[unitmap.idx_b],
        ),
    }
    return pd.DataFrame(dmap)

In [6]:
from bloodmoon.mask import count, variance

def get_varmap(camera: CodedMaskCamera, sdl: DataLoader) -> NDArray:
    detector = count(camera, sdl.DLdata)[0]
    varmap = variance(camera, detector)
    return varmap


varmapA, varmapB = map(lambda sdl: get_varmap(wfm, sdl), (sdlA, sdlB))

In [7]:
ds.pixels_angular_resolution(wfm)
cu_map = mgm.get_srcmap_for_unit(logA.log['ID'], logB.log['ID'])

# Table - CAMERA A
data_camA = gather_cam_data(logA, catA, sdlA, wfm, varmapA)

# Table - CAMERA B
data_camB = gather_cam_data(logB, catB, sdlB, wfm, varmapB)


Pixel angular resolution at upscaling (x, y): (5, 1)
  - fine direction: 0.8465 arcmin
  - coarse direction: 8.4653 arcmin



Analysing SCOX1:   0%|          | 0/25 [00:00<?, ?it/s]WARNING: The following header keyword is invalid or follows an unrecognized non-standard convention:
CATALOG =RXTE-ASM_BeppoSAX-WFC_catalog_2-50keV.fits / Catalog file               [astropy.io.fits.card]
Analysing SCOX1:   0%|          | 0/25 [00:00<?, ?it/s]WARNING: The following header keyword is invalid or follows an unrecognized non-standard convention:
CATALOG =RXTE-ASM_BeppoSAX-WFC_catalog_2-50keV.fits / Catalog file               [astropy.io.fits.card]
Analysing LEMX-CAM1BS3: 100%|██████████| 25/25 [00:03<00:00,  7.82it/s]


In [8]:
unit_data = get_joint_tab(data_camA, data_camB, cu_map)

KWS = {
    'label': 'Table1',
    'caption': 'Testing $`to\\_latex`$ fn.',
    'float_format': "%.1f",
    'column_format': 'l' + 'c' * (len(unit_data.columns) - 2) + 'r',
}
tab = mgm.df2TeXtab(
    df=sort_by(unit_data, 'SNR'),
    adjust_tabfrmt=adjust_Tabfrmt,
    save_to=f'{OUT_RESULTS_PATH}/../texTable_Unit_results_{DATASET}_{E_min}-{E_max}.tex',
    overwrite=True,
    **KWS,
)

In [9]:
unit_data.sort_values('SNR', ascending=False, ignore_index=True)

,Source,DthetaX_A,DthetaY_A,TrueCts_A,ReconstrCts_A,Dcts_A,Dcts_A_var,DthetaX_B,DthetaY_B,TrueCts_B,ReconstrCts_B,Dcts_B,Dcts_B_var,SNR
0,SCOX1,0.088250,0.623922,1026162.0,1.013612e+06,-12.388758,-9.147905,0.140636,0.538394,936589.0,918798.201673,-18.383191,-13.165622,1009.777627
1,GX5-1,0.020263,-0.308112,114793.0,1.152621e+05,1.384582,0.341950,-0.024036,-1.785391,115243.0,117169.675086,5.675462,1.425786,107.040846
2,GX349+2,0.322842,-5.398838,81908.0,9.155182e+04,33.696598,7.029673,0.040995,-1.088339,82908.0,83417.938676,1.771005,0.377367,79.132400
3,GX17+2,0.206212,-2.476350,71902.0,6.992776e+04,-7.362576,-1.439084,-0.014584,-3.033534,76199.0,75409.534803,-2.859949,-0.584223,68.407794
4,GX9+1,0.184955,2.858255,63221.0,6.747390e+04,16.914318,3.100067,-0.008232,0.261123,63488.0,64921.940901,5.690959,1.061151,60.217922
5,GX340+0,-0.223318,6.510455,45655.0,3.993396e+04,-26.775060,-4.170235,-0.006237,0.942354,48686.0,49880.909853,5.415432,0.884262,44.481595
6,GX13+1,0.173172,-0.938934,36408.0,3.919204e+04,14.590724,2.029370,-0.020011,-1.912514,36790.0,41794.128735,26.089369,3.703177,36.845125
7,GX3+1,0.032839,-1.723854,35947.0,4.401055e+04,42.529949,5.877763,0.218943,3.310623,36797.0,35518.419392,-6.665334,-0.946181,35.500927
8,X1820-303,-0.063129,4.717723,34408.0,3.842284e+04,21.644041,2.926537,-0.148287,-4.541084,33805.0,34889.557626,5.898780,0.802599,33.357892
9,CIRX1,-0.640749,8.230652,20698.0,1.900709e+04,-11.753236,-1.232558,0.304927,4.517315,22400.0,22484.951342,0.567605,0.062866,32.341236
